In [ ]:
# --- Standard Library ---
import os
import json

import pandas as pd
import numpy as np
 
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
 
from transformers import BertModel, BertTokenizer

from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

In [82]:
CSV_PATH    = "fashion_data.csv"   # your synthetic dataset
MODEL_DIR   = "./fashion-bert"     # where to save the model
BERT_NAME   = "bert-base-uncased"
MAX_LEN     = 64
BATCH_SIZE  = 20
EPOCHS      = 25
LR          = 1e-5
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
 

In [83]:
LABEL_MAPS = {
    "occasion": {
        "casual": 0,
        "date": 1,
        "office": 2,
        "party": 3,
        "wedding": 4
    },

    "formality": {
        "casual": 0,
        "formal": 1,
        "semi-formal": 2
    },

    "constraint": {
        "bold": 0,
        "no_constraint": 1,
        "understated": 2
        
    },

    "color_tone": {
        "bright": 0,
        "dark": 1,
        "neutral": 2,
        "soft": 3
    }
}

In [84]:
REVERSE_MAPS = {cat: {v: k for k, v in m.items()} for cat, m in LABEL_MAPS.items()}

In [85]:
class FashionDataset(Dataset):
    """
    Expects a CSV with columns:
        prompt, occasion, formality, constraint, color_tone
    """
 
    def __init__(self, df: pd.DataFrame, tokenizer: BertTokenizer):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        tokens = self.tokenizer(
            row["prompt"],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
 
        return {
            "input_ids":        tokens["input_ids"].squeeze(),       # (MAX_LEN,)
            "attention_mask":   tokens["attention_mask"].squeeze(),   # (MAX_LEN,)
            "occasion_label":   torch.tensor(LABEL_MAPS["occasion"][row["occasion"]],   dtype=torch.long),
            "formality_label":  torch.tensor(LABEL_MAPS["formality"][row["formality"]], dtype=torch.long),
            "constraint_label": torch.tensor(LABEL_MAPS["constraint"][row["constraint"]],dtype=torch.long),
            "color_label":      torch.tensor(LABEL_MAPS["color_tone"][row["color_tone"]],dtype=torch.long),
        }

In [ ]:
class FashionIntentModel(nn.Module):
    """
    BERT backbone with 4 independent classification heads.
    Each head predicts one intent category independently.
    """
 
    def __init__(self):
    super().__init__()
    # Load base BERT from HuggingFace, not your local folder
    self.bert = BertModel.from_pretrained("bert-base-uncased")  # ← change this
    hidden = self.bert.config.hidden_size

    self.occasion_head   = nn.Linear(hidden, len(LABEL_MAPS["occasion"]))
    self.formality_head  = nn.Linear(hidden, len(LABEL_MAPS["formality"]))
    self.constraint_head = nn.Linear(hidden, len(LABEL_MAPS["constraint"]))
    self.color_head      = nn.Linear(hidden, len(LABEL_MAPS["color_tone"]))
    self.dropout         = nn.Dropout(0.3)
 
    def forward(self, input_ids, attention_mask):
        output    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = output.last_hidden_state[:, 0, :]  # CLS token → (batch, 768)
        cls_token = self.dropout(cls_token)
 
        return {
            "occasion":   self.occasion_head(cls_token),    # (batch, 5)
            "formality":  self.formality_head(cls_token),   # (batch, 3)
            "constraint": self.constraint_head(cls_token),  # (batch, 3)
            "color":      self.color_head(cls_token),       # (batch, 4)
        }

In [87]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0.0
 
    for batch in dataloader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
 
        occasion_labels   = batch["occasion_label"].to(DEVICE)
        formality_labels  = batch["formality_label"].to(DEVICE)
        constraint_labels = batch["constraint_label"].to(DEVICE)
        color_labels      = batch["color_label"].to(DEVICE)
 
        # Forward pass
        outputs = model(input_ids, attention_mask)
 
        # Sum loss across all four heads
        loss = (
            criterion(outputs["occasion"],   occasion_labels)   +
            criterion(outputs["formality"],  formality_labels)  +
            criterion(outputs["constraint"], constraint_labels) +
            criterion(outputs["color"],      color_labels)
        )
 
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        total_loss += loss.item()
 
    return total_loss / len(dataloader)

In [89]:
def evaluate(model, dataloader):
    model.eval()
 
    all_preds  = {cat: [] for cat in LABEL_MAPS}
    all_labels = {cat: [] for cat in LABEL_MAPS}
 
    # Map from output key → dataset key (output uses "color", dataset uses "color_tone")
    key_map = {
        "occasion":   "occasion_label",
        "formality":  "formality_label",
        "constraint": "constraint_label",
        "color":      "color_label",
    }
 
    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            outputs        = model(input_ids, attention_mask)
 
            for out_key, label_key in key_map.items():
                preds  = outputs[out_key].argmax(dim=1).cpu().tolist()
                labels = batch[label_key].tolist()
 
                # Map output key back to LABEL_MAPS key
                cat = out_key if out_key != "color" else "color_tone"
                all_preds[cat].extend(preds)
                all_labels[cat].extend(labels)
 
    print("\n===== Evaluation Results =====")
    for cat in LABEL_MAPS:
        f1 = f1_score(all_labels[cat], all_preds[cat], average="weighted")
        print(f"\n[{cat.upper()}]  Weighted F1: {f1:.4f}")
        print(classification_report(
            all_labels[cat],
            all_preds[cat],
            target_names=list(LABEL_MAPS[cat].keys())
        ))
 

In [ ]:
def predict(prompt: str, model: FashionIntentModel, tokenizer: BertTokenizer) -> dict:
    model.eval()
 
    tokens = tokenizer(
        prompt,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
 
    input_ids      = tokens["input_ids"].to(DEVICE)
    attention_mask = tokens["attention_mask"].to(DEVICE)
 
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
 
    result = {
        "occasion":   REVERSE_MAPS["occasion"][outputs["occasion"].argmax().item()],
        "formality":  REVERSE_MAPS["formality"][outputs["formality"].argmax().item()],
        "constraint": REVERSE_MAPS["constraint"][outputs["constraint"].argmax().item()],
        "color_tone": REVERSE_MAPS["color_tone"][outputs["color"].argmax().item()],
    }
 
    # Also return confidence scores for each category
    result["scores"] = {
        "occasion":   outputs["occasion"].softmax(dim=1).cpu().tolist()[0],
        "formality":  outputs["formality"].softmax(dim=1).cpu().tolist()[0],
        "constraint": outputs["constraint"].softmax(dim=1).cpu().tolist()[0],
        "color_tone": outputs["color"].softmax(dim=1).cpu().tolist()[0],
    }
 
    return result
 

In [91]:
def predict(prompt: str, model: FashionIntentModel, tokenizer: BertTokenizer) -> dict:
    model.eval()
 
    tokens = tokenizer(
        prompt,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
 
    input_ids      = tokens["input_ids"].to(DEVICE)
    attention_mask = tokens["attention_mask"].to(DEVICE)
 
    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
 
    result = {
        "occasion":   REVERSE_MAPS["occasion"][outputs["occasion"].argmax().item()],
        "formality":  REVERSE_MAPS["formality"][outputs["formality"].argmax().item()],
        "constraint": REVERSE_MAPS["constraint"][outputs["constraint"].argmax().item()],
        "color_tone": REVERSE_MAPS["color_tone"][outputs["color"].argmax().item()],
    }
 
    # Also return confidence scores for each category
    result["scores"] = {
        "occasion":   outputs["occasion"].softmax(dim=1).cpu().tolist()[0],
        "formality":  outputs["formality"].softmax(dim=1).cpu().tolist()[0],
        "constraint": outputs["constraint"].softmax(dim=1).cpu().tolist()[0],
        "color_tone": outputs["color"].softmax(dim=1).cpu().tolist()[0],
    }
    print(f"\n[Inference] Prompt: {prompt}")
    print(f"Predicted Intent: {result}")
    return result


In [92]:
def save_model(model, tokenizer, path=MODEL_DIR):
    os.makedirs(path, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(path, "model.pt"))
    tokenizer.save_pretrained(path)
    with open(os.path.join(path, "label_maps.json"), "w") as f:
        json.dump(LABEL_MAPS, f)
    print(f"Model saved to {path}")
 
 
def load_model(path=MODEL_DIR):
    tokenizer = BertTokenizer.from_pretrained(path)
    model     = FashionIntentModel().to(DEVICE)
    model.load_state_dict(torch.load(os.path.join(path, "model.pt"), map_location=DEVICE))
    print(f"Model loaded from {path}")
    return model, tokenizer

In [93]:
if __name__ == "__main__":
 
    # --- Load Data ---
    print("Loading data...")
    df = pd.read_csv(CSV_PATH)
 
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
    print(f"Train: {len(train_df)} | Val: {len(val_df)}")
 
    # --- Tokenizer ---
    tokenizer = BertTokenizer.from_pretrained(BERT_NAME)
 
    # --- Datasets & Dataloaders ---
    train_dataset = FashionDataset(train_df, tokenizer)
    val_dataset   = FashionDataset(val_df,   tokenizer)
 
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
 
    # --- Model, Loss, Optimizer ---
    model     = FashionIntentModel().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=LR)
 
    print(f"\nTraining on: {DEVICE}")
    print(f"Parameters : {sum(p.numel() for p in model.parameters()):,}\n")
 
    # --- Training ---
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, criterion)
        print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {loss:.4f}")
 
    # --- Evaluation ---
    evaluate(model, val_loader)
 
    # --- Save ---
    save_model(model, tokenizer)
 
    # --- Quick Inference Test ---
    print("\n===== Inference Test =====")
    test_prompts = [
        "i have to go to a club tomorrow night, i want a classy look but not too revealing",
        "job interview tomorrow, need to look professional but not over",
        "beach vacation with my girls, something fun and colorful",
        "its my birthday, i need to look outstanding",
    ]
 
    for prompt in test_prompts:
        result = predict(prompt, model, tokenizer)
        print(f"\nPrompt   : {prompt}")
        print(f"Occasion : {result['occasion']}")
        print(f"Formality: {result['formality']}")
        print(f"Constraint:{result['constraint']}")
        print(f"Color    : {result['color_tone']}")

Loading data...
Train: 632 | Val: 159


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2460.71it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Training on: cpu
Parameters : 109,493,775

Epoch 1/25 | Loss: 5.2322
Epoch 2/25 | Loss: 4.7650
Epoch 3/25 | Loss: 3.8537
Epoch 4/25 | Loss: 3.1390
Epoch 5/25 | Loss: 2.6661
Epoch 6/25 | Loss: 2.3728
Epoch 7/25 | Loss: 2.0950
Epoch 8/25 | Loss: 1.8814
Epoch 9/25 | Loss: 1.6838
Epoch 10/25 | Loss: 1.5374
Epoch 11/25 | Loss: 1.3859
Epoch 12/25 | Loss: 1.2608
Epoch 13/25 | Loss: 1.0934
Epoch 14/25 | Loss: 0.9674
Epoch 15/25 | Loss: 0.8564
Epoch 16/25 | Loss: 0.7581
Epoch 17/25 | Loss: 0.6423
Epoch 18/25 | Loss: 0.5790
Epoch 19/25 | Loss: 0.4899
Epoch 20/25 | Loss: 0.4280
Epoch 21/25 | Loss: 0.3908
Epoch 22/25 | Loss: 0.3441
Epoch 23/25 | Loss: 0.3087
Epoch 24/25 | Loss: 0.2867
Epoch 25/25 | Loss: 0.2620

===== Evaluation Results =====

[OCCASION]  Weighted F1: 0.9748
              precision    recall  f1-score   support

      casual       0.97      0.94      0.95        32
        date       1.00      0.97      0.98        31
      office       0.97      1.00      0.98        32
       p